# Cross-Platform Topic Matching (BTM)
<!-- structured-notebook -->
## Notebook Summary
Purpose: align topics across BERTopic models trained on different platforms (e.g. PubMed ↔ Reddit) to produce cross-platform topic pairs that feed downstream diffusion, sentiment, and vocabulary-shift analyses.

Main steps:
- Load two BERTopic models and their document corpora.
- Cross-assign each corpus's documents using the other model's topic space.
- Produce the pairing strength matrix `S` whose rows sum to ≤ 1 (plus a -1 outlier column).
- Compute corpus-level diagnostics (C, Cw, θ, U, SA, A) for sanity checking.
- Hand the S matrices to `02_finalizing_pairs.ipynb` for mutual-top-1 selection.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/` | Produced by Reddit preprocessing |
| Input | `<DATA_ROOT>/PubMed/output/bertopic/<variant>/` | Produced by PubMed `03_topic_modeling/01_data_analysis_all_mpnet.ipynb` |
| Input | `<DATA_ROOT>/Telegram/output/bertopic/` | Produced by Telegram `03_topic_modeling/01_topic_modeling.ipynb` |
| Output | `<DATA_ROOT>/topic_matchings/<pair>/BTM_S_*.csv` | Consumed by `02_finalizing_pairs.ipynb` |


# BTM Metrics
<!-- structured-notebook -->
## Summary
All metrics derive from the S-matrix — the share of documents from native topic *i* cross-assigned to topic *j* in the other model's topic space.

| Symbol | Name | Definition | Interpretation |
|---|---|---|---|
| S | Pairing strength | Docs assigned to both native topic *i* and cross topic *j*, divided by docs in *i*. Always ≤ 1. | Primary metric for picking pairs. High S = strong thematic link. |
| Topic uniqueness | — | Special case of S against the -1 outlier column. ≥ 0.5 → "unique to its own corpus". | Flags topics with no analogue in the other corpus. |
| C | Corpus closeness | Thematic overlap between the two corpora. `C_B` / `Cw_B` measure how well model A covers B's topics (or docs, weighted). | Sanity check. |
| Cw | Weighted corpus closeness | Same as C, weighted by native topic size. | Complements C. |
| θ | Size-weighting signal | θ ∈ [−1, 1]. ≈ 1: closeness driven by larger native topics. ≈ 0: size-independent. ≈ −1: driven by smaller topics. | Diagnostic — shows whether closeness is dominated by a few large topics or spread evenly. |
| U, Uw | Corpus uniqueness | Alternative view of C — how independent the corpora are. | Sanity check. |
| SA | Topic alignment strength | Max S across all cross topics (excluding -1) for a native topic. | Whether a topic maps to a single theme or is scattered. |
| A, Aw | Corpus alignment | Average SA across all native topics (weighted and unweighted). | Whether pair relationships are focused or smeared. |

**Pair-selection rule.** Good pair → high S in both directions. The other metrics are sanity checks. **Always log θ** — if θ ≈ 1 and the corpora look "close," the closeness is really being driven by a few large topics, not a broad thematic overlap. A 1:3 corpus-size ratio is not a problem for the algorithm.


# Required Inputs per Model
<!-- structured-notebook -->
## Summary

For each platform whose topics will be paired, prepare:

1. **Preprocessed docs** — the exact text that was trained on.
   - `preprocessed-docs.pkl` → `List[str]` of length `N_docs`.
   - If dedup was applied: also `unique_docs.pkl`; BTM uses the **unique** docs.

2. **Trained BERTopic model.**
   - Preferred: saved without embedder as `bertopic_no_embed/` (with `prediction_data=True` in HDBSCAN), plus the embedder path/name used at training.
   - Acceptable: saved with embedder as `bertopic_base.model`.

3. **Native topic labels (hard).**
   - For unique docs: `train_topics_unique.npy` (length U).
   - For kept docs: `train_topics_original.npy` (length K).
   - If missing, recompute via `model.transform(docs, embeddings=...)`.

4. **Normalization policy.**
   - A probe-embeddings file lets you auto-detect L2 normalization.
   - Or set an explicit flag on the normalization call.

### Strongly recommended optional inputs

- `topics.csv` (from `get_topic_info()`) for readable labels
- `map_orig_to_unique.npy` + `dup_sizes` weights so S reflects original frequency
- Per-doc UTC timestamps for downstream temporal analyses
- Doc-topic probabilities for a soft-assignment BTM variant


# Result: 17 Core Cross-Platform Topics
<!-- structured-notebook -->
## Summary
Running BTM across all platform pairs produced 17 core topics matched across science, news, and social media. See the paper §3.5 for the full list:

Centenarians, Senescent Cells, Telomeres, Rapamycin, Metformin, Dementia, Physical Activity, Glycemic Control, Gut Microbiome, Epigenetics, Mitochondria, Sirtuins, Autophagy, Cardiovascular Disease, Obesity, Cancer, Modafinil & Nootropics.

Per-pair output folders live under `<DATA_ROOT>/Reddit/output/topic_matching/` (e.g. `pubmed_mpnet-reddit/`, `reddit-telegram/`, `multi-platform/`).


## Setup & Imports

In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_MODELS,
)
import sys
from pathlib import Path
# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

In [ ]:
import os
import pickle
from collections import Counter
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Union

import numpy as np
import pandas as pd
from scipy import sparse

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

## Configuration

Define the two corpora to compare. Each `CorpusSpec` bundles the model path, embedder identifier, document path, and optional precomputed assets (embeddings, native topic labels). Adjust these paths for whichever platform pair you are comparing.

In [ ]:
@dataclass
class CorpusSpec:
    """Configuration for one corpus/model."""
    name: str
    model_path: str
    embedder: Optional[str]                          # SentenceTransformer model name or path
    docs_path: str
    docs_text_column: Optional[str] = None           # column name if docs are in CSV
    train_embeddings_path: Optional[str] = None      # for unit-norm detection
    native_topics_path: Optional[str] = None         # per-doc topic IDs to skip native assignment
    embeddings_path: Optional[str] = None            # precomputed embeddings (same order as docs)
    embedding_model_on_load: Optional[str] = None    # forwarded to BERTopic.load()


@dataclass
class RunSpec:
    """Run-level configuration."""
    output_dir: str
    batch_size: int = 256
    device: Optional[str] = None       # "cuda:0", "mps", or None for CPU
    top_k_matches: int = 3
    make_output_dir: bool = True
    sample_n: Optional[int] = None     # set e.g. 1000 for a quick test
    prefer_similarity: bool = True     # BTM-style portable assignment (recommended)

In [ ]:
# --- Corpus A: e.g., YouTube ---
corpus_A = CorpusSpec(
    name="youtube",
    model_path=REDDIT_MODELS / "round_11" / "mpnet_topic_model_no_umap",
    embedder="all-mpnet-base-v2",
    docs_path=REDDIT_MODELS / "round_11" / "preprocessed_data.pkl",
    train_embeddings_path=REDDIT_MODELS / "round_11" / "embeddings.npy",
    native_topics_path=REDDIT_MODELS / "round_11" / "text_topic.csv",
    embeddings_path=REDDIT_MODELS / "round_11" / "embeddings.npy",
    embedding_model_on_load=None,
)

# --- Corpus B: e.g., Reddit ---
corpus_B = CorpusSpec(
    name="reddit",
    model_path="../scripts/topic-matching/topic_modelling_v2/round_10/bertopic_no_embed",
    embedder="../scripts/topic-matching/topic_modelling_v2/round_10/embedder",
    docs_path="../scripts/topic-matching/topic_modelling_v2/round_4/unique_docs.pkl",
    train_embeddings_path="../scripts/topic-matching/topic_modelling_v2/round_10/embeddings_fp32_l2.npy",
    native_topics_path="../scripts/topic-matching/topic_modelling_v2/round_10/train_topics_unique.npy",
    embeddings_path="../scripts/topic-matching/topic_modelling_v2/round_10/embeddings_fp32_l2.npy",
    embedding_model_on_load=None,
)

# --- Run settings ---
run = RunSpec(
    output_dir="../scripts/topic-matching/reddit-youtube/round_2",
    batch_size=256,
    device=None,
    top_k_matches=3,
    make_output_dir=True,
    sample_n=None,             # set to e.g. 1000 for a quick test run
    prefer_similarity=True,
)

## Utility Functions

Helper functions for embedding normalisation, unit-norm detection, and topic order inference.

In [ ]:
def _as_float32(E):
    E = np.asarray(E)
    if E.dtype != np.float32:
        E = E.astype(np.float32, copy=False)
    return E


def _unit_norm_rows(X):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(n, 1e-12, None)


def is_unit_norm(E, tol=1e-3):
    """Check whether embeddings are already L2-normalised."""
    norms = np.linalg.norm(_as_float32(E), axis=1)
    return abs(np.median(norms) - 1.0) <= tol


def ensure_unit_norm(E):
    E = _as_float32(E)
    n = np.linalg.norm(E, axis=1, keepdims=True)
    return E / np.clip(n, 1e-12, None)


def detect_training_policy_from_file(path, sample=2000):
    """Detect whether saved embeddings were L2-normalised during training."""
    E = np.load(path, mmap_mode="r")
    n = min(sample, len(E))
    return is_unit_norm(E[:n])


def _has_umap_and_cluster(model) -> bool:
    has_umap = getattr(model, "umap_model", None) is not None
    has_hdb = (
        getattr(model, "hdbscan_model", None) is not None
        or getattr(model, "cluster_model", None) is not None
    )
    return has_umap and has_hdb

## Topic Order Helpers

BERTopic stores topic representations in multiple forms (c-TF-IDF matrix, topic embeddings). The row ordering may not match the topic IDs directly. These helpers recover the correct mapping.

In [ ]:
def _topics_order_for_ctfidf(model):
    """Return topic IDs whose order matches rows of model.c_tf_idf_."""
    tm = getattr(model, "topic_mapper_", None)
    if tm is not None:
        topic_ids = getattr(tm, "topic_ids", None)
        if topic_ids is None and hasattr(tm, "__dict__"):
            topic_ids = tm.__dict__.get("topic_ids", None)
        if topic_ids:
            return [t for t in topic_ids if t != -1]
        for attr in ("map_index_to_id", "index_to_id", "_index_to_id"):
            map_it = getattr(tm, attr, None)
            if isinstance(map_it, (list, tuple)):
                return [t for t in map_it if t != -1]

    topics_dict = getattr(model, "get_topics", lambda: {})()
    if isinstance(topics_dict, dict) and topics_dict:
        cand = sorted([t for t in topics_dict.keys() if t != -1])
        ctf = getattr(model, "c_tf_idf_", None)
        if ctf is not None and len(cand) == ctf.shape[0]:
            return cand

    info = model.get_topic_info()
    return info.loc[info["Topic"] != -1, "Topic"].tolist()


def _topics_order_for_topic_embeddings(model):
    """Return topic IDs aligned with rows of model.topic_embeddings_."""
    Te = getattr(model, "topic_embeddings_", None)
    if Te is None:
        raise RuntimeError("topic_embeddings_ not available")
    k = len(Te)

    info = model.get_topic_info()
    topic_ids = info.loc[info["Topic"] != -1, "Topic"].tolist()
    if len(topic_ids) == k:
        return topic_ids

    topics_dict = getattr(model, "get_topics", lambda: {})()
    if isinstance(topics_dict, dict) and topics_dict:
        cand = sorted([t for t in topics_dict.keys() if t != -1])
        if len(cand) == k:
            return cand

    return list(range(k))  # synthetic IDs aligned to rows

## Load Models and Documents

Load both BERTopic models and their document corpora. Documents can be stored as pickle (list of strings) or CSV files.

In [ ]:
def load_docs(path: str, text_col: Optional[str] = None, sample_n: Optional[int] = None):
    """Load documents from pickle or CSV."""
    lower = path.lower()
    if lower.endswith((".pkl", ".pickle")):
        with open(path, "rb") as f:
            docs = pickle.load(f)
        docs = [str(x).strip() for x in docs if isinstance(x, str) or pd.notna(x)]
        docs = [d for d in docs if d]
        return docs[:sample_n] if sample_n else docs

    if lower.endswith(".csv"):
        df = pd.read_csv(path)
        col = text_col if (text_col and text_col in df.columns) else (
            "text" if "text" in df.columns else
            next((c for c in df.columns if df[c].dtype == "object"), df.columns[0])
        )
        s = df[col].astype(str).str.strip()
        s = s[s.notna() & (s != "")]
        docs = s.tolist()
        return docs[:sample_n] if sample_n else docs

    raise ValueError(f"Unsupported docs file type: {path}")


def load_embeddings(path: Optional[str], n_docs: Optional[int], unit_norm: bool) -> Optional[np.ndarray]:
    """Load and optionally L2-normalise embeddings."""
    if not path:
        return None
    E = np.load(path)
    if n_docs is not None and E.shape[0] != n_docs:
        raise ValueError(f"Embeddings rows ({E.shape[0]}) != n_docs ({n_docs}) for {path}")
    E = _as_float32(E)
    return _unit_norm_rows(E) if unit_norm else E


def load_model(spec: CorpusSpec) -> BERTopic:
    """Load a BERTopic model from a CorpusSpec."""
    return BERTopic.load(spec.model_path, embedding_model=spec.embedding_model_on_load)


def load_topic_ids(path: str, n_docs: Optional[int] = None) -> np.ndarray:
    """Load per-document topic IDs from npy, csv, or parquet."""
    lower = path.lower()

    if lower.endswith((".npy", ".npz")):
        arr = np.load(path)
        arr = np.ravel(arr).astype(np.int32)
        return arr[:n_docs] if n_docs is not None else arr

    if lower.endswith(".csv"):
        df = pd.read_csv(path)
        if "topic" in df.columns:
            s = df["topic"]
        else:
            num_cols = [
                c for c in df.columns
                if pd.api.types.is_integer_dtype(df[c]) or pd.api.types.is_numeric_dtype(df[c])
            ]
            s = df[num_cols[0]] if num_cols else df.iloc[:, -1]
        arr = s.to_numpy(dtype=np.int32)
        return arr[:n_docs] if n_docs is not None else arr

    if lower.endswith(".parquet"):
        df = pd.read_parquet(path)
        cols = {c.lower(): c for c in df.columns}
        doc_col = cols.get("doc_idx") or cols.get("doc_id") or cols.get("index")
        topic_col = cols.get("topic_id") or cols.get("topic") or cols.get("label")
        if doc_col is None or topic_col is None:
            raise ValueError(
                f"Parquet at {path} must contain 'doc_idx' and 'topic_id' columns; "
                f"found: {list(df.columns)}"
            )
        df = df[[doc_col, topic_col]].dropna().sort_values(doc_col)
        if n_docs is None:
            return df[topic_col].to_numpy(dtype=np.int32)
        topics = np.full((n_docs,), -1, dtype=np.int32)
        idx = df[doc_col].astype(int).to_numpy()
        mask = (idx >= 0) & (idx < n_docs)
        topics[idx[mask]] = df[topic_col].astype(np.int32).to_numpy()[mask]
        return topics

    raise ValueError(f"Unsupported topics file type: {path}")

In [ ]:
# Load models
model_A = load_model(corpus_A)
model_B = load_model(corpus_B)

# Load documents
docs_A = load_docs(corpus_A.docs_path, text_col=corpus_A.docs_text_column, sample_n=run.sample_n)
docs_B = load_docs(corpus_B.docs_path, text_col=corpus_B.docs_text_column, sample_n=run.sample_n)

print(f"Corpus A ({corpus_A.name}): {len(docs_A):,} documents, "
      f"{len(model_A.get_topics()) - 1} topics")
print(f"Corpus B ({corpus_B.name}): {len(docs_B):,} documents, "
      f"{len(model_B.get_topics()) - 1} topics")

## Cross-Assignment Strategy

The cross-assignment works as follows:

1. **Native assignment**: For each corpus, obtain the topic label each document was originally assigned to (either from a saved file or by re-running the model).
2. **Cross assignment**: Assign documents from corpus A using model B's topic space (and vice versa). This tells us which "foreign" topic each document maps to.

The assignment method is chosen automatically:
- **Topic embeddings (preferred)**: Compute cosine similarity between each document embedding and each topic embedding. The topic with highest similarity wins. This is portable and does not require UMAP/HDBSCAN.
- **c-TF-IDF fallback**: If topic embeddings are unavailable, vectorise documents using the model's fitted vectoriser and compute similarity against the c-TF-IDF matrix.

Embeddings are L2-normalised if the training embeddings were normalised (auto-detected from a sample).

In [ ]:
def _batched_encode(embedder_id_or_dir, docs, want_unit_norm=True, batch_size=256, device=None):
    """Encode documents in batches using a SentenceTransformer."""
    st = SentenceTransformer(embedder_id_or_dir, device=device)
    try:
        st.max_seq_length = 256
    except Exception:
        pass
    n = len(docs)
    for i in range(0, n, batch_size):
        j = min(i + batch_size, n)
        E = st.encode(
            docs[i:j],
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=want_unit_norm,
            show_progress_bar=False,
        ).astype(np.float32)
        if want_unit_norm:
            E = _unit_norm_rows(E)
        yield i, j, E


def _cosine_argmax(E_docs: np.ndarray, E_topics: np.ndarray) -> np.ndarray:
    """For unit-norm inputs, cosine similarity = dot product."""
    S = E_docs @ E_topics.T
    return np.argmax(S, axis=1).astype(np.int32)

In [ ]:
def _assign_via_topic_embeddings(
    model, docs, embedder, want_unit_norm=True, batch_size=256, device=None,
    E_docs: Optional[np.ndarray] = None,
):
    """Assign documents to topics using topic embedding cosine similarity."""
    Te = getattr(model, "topic_embeddings_", None)
    if Te is None:
        raise RuntimeError("topic_embeddings_ not available on model")

    Te = _unit_norm_rows(_as_float32(Te))
    topic_ids = _topics_order_for_topic_embeddings(model)
    if len(topic_ids) != Te.shape[0] and topic_ids != list(range(Te.shape[0])):
        raise RuntimeError(
            f"topic_embeddings_ rows ({Te.shape[0]}) != topic_ids ({len(topic_ids)})"
        )

    if E_docs is not None:
        E_docs = _unit_norm_rows(_as_float32(E_docs)) if want_unit_norm else _as_float32(E_docs)
        idx = _cosine_argmax(E_docs, Te)
        return np.asarray([topic_ids[k] for k in idx], dtype=np.int32)

    out = np.empty(len(docs), dtype=np.int32)
    for i, j, E in _batched_encode(embedder, docs, want_unit_norm, batch_size, device):
        idx = _cosine_argmax(E, Te)
        out[i:j] = np.asarray([topic_ids[k] for k in idx], dtype=np.int32)
    return out


def _assign_via_ctfidf(model, docs, batch_size=4096):
    """Fallback assignment using c-TF-IDF similarity."""
    vec = model.vectorizer_model
    Ct = model.c_tf_idf_
    if Ct is None or vec is None:
        raise RuntimeError("Model lacks c_tf_idf_ or vectorizer_model.")
    if not sparse.issparse(Ct):
        Ct = sparse.csr_matrix(Ct)

    topics_order = _topics_order_for_ctfidf(model)
    n_rows = Ct.shape[0]
    if len(topics_order) != n_rows:
        print(
            f"[WARN] c_tf_idf_ rows ({n_rows}) != derived topic ids "
            f"({len(topics_order)}); using indices 0..{n_rows - 1}"
        )
        topics_order = list(range(n_rows))

    CtT = Ct.T  # (vocab, n_rows)
    n = len(docs)
    out = np.empty(n, dtype=np.int32)

    for i in range(0, n, batch_size):
        j = min(i + batch_size, n)
        X = vec.transform(docs[i:j])   # (m, vocab)
        S = X @ CtT                    # (m, n_rows)
        if sparse.issparse(S):
            S = S.toarray()

        empty = (X.getnnz(axis=1) == 0) if sparse.issparse(X) else (X.sum(axis=1) == 0)
        best_idx = np.argmax(S, axis=1).astype(np.int32)
        mapped = np.take(np.asarray(topics_order, dtype=np.int32), best_idx, mode="clip")
        if np.ndim(empty) > 1:
            empty = np.asarray(empty).ravel()
        mapped = np.where(empty, -1, mapped)
        out[i:j] = mapped

    return out


def _safe_assign(
    model, docs, embedder, want_unit_norm=True, batch_size=256, device=None,
    E_docs: Optional[np.ndarray] = None, prefer_similarity: bool = True,
):
    """
    Unified assignment with automatic fallback chain:
      topic_embeddings -> c-TF-IDF
    """
    if prefer_similarity or not _has_umap_and_cluster(model):
        try:
            return _assign_via_topic_embeddings(
                model, docs, embedder, want_unit_norm, batch_size, device, E_docs
            )
        except Exception as e_te:
            print(f"[INFO] topic_embeddings_ fallback unavailable: {e_te} -> c-TF-IDF")
            return _assign_via_ctfidf(model, docs, batch_size=max(2048, batch_size))

    # Native path (when UMAP + HDBSCAN are available)
    if E_docs is not None:
        try:
            E_docs = (
                _unit_norm_rows(_as_float32(E_docs)) if want_unit_norm
                else _as_float32(E_docs)
            )
            out = np.empty(len(docs), dtype=np.int32)
            for i in range(0, len(docs), batch_size):
                j = min(i + batch_size, len(docs))
                topics_chunk, _ = model.transform(docs[i:j], embeddings=E_docs[i:j])
                out[i:j] = np.asarray(topics_chunk, dtype=np.int32)
            return out
        except Exception as e:
            print(f"[INFO] .transform(precomputed E) failed: {e} -> falling back")

    try:
        out = np.empty(len(docs), dtype=np.int32)
        for i, j, E in _batched_encode(embedder, docs, want_unit_norm, batch_size, device):
            topics_chunk, _ = model.transform(docs[i:j], embeddings=E)
            out[i:j] = np.asarray(topics_chunk, dtype=np.int32)
        return out
    except Exception as e:
        print(f"[INFO] .transform() failed: {e} -> falling back to similarity")

    try:
        return _assign_via_topic_embeddings(
            model, docs, embedder, want_unit_norm, batch_size, device, E_docs
        )
    except Exception as e_te:
        print(f"[INFO] topic_embeddings_ fallback unavailable: {e_te} -> c-TF-IDF")
    return _assign_via_ctfidf(model, docs, batch_size=max(2048, batch_size))

## Compute Similarity Matrix

The core cross-assignment function assigns native and cross topics for both corpora, then computes the pairing strength matrix $S$.

For corpus A with $T_A$ native topics and corpus B with $T_B$ topics, $S_A$ is a $(T_A \times T_B)$ matrix where:

$$
S_A[i, j] = \frac{|\{d \in A : \text{native}(d) = i \land \text{cross}(d) = j\}|}{|\{d \in A : \text{native}(d) = i\}|}
$$

In [ ]:
def cross_assign_bt(
    model_A, docs_A, embedder_A, train_emb_path_A=None, native_topics_A_path=None, embeddings_A_path=None,
    model_B=None, docs_B=None, embedder_B=None, train_emb_path_B=None, native_topics_B_path=None, embeddings_B_path=None,
    batch_size=256, device=None, prefer_similarity=True,
):
    """
    Cross-assign documents between two models.
    Returns (A_native, A_cross, B_native, B_cross), all int32 ndarrays.
    """
    want_unit_A = True if train_emb_path_A is None else detect_training_policy_from_file(train_emb_path_A)
    want_unit_B = True if train_emb_path_B is None else detect_training_policy_from_file(train_emb_path_B)

    # --- Native A ---
    if native_topics_A_path:
        print(f"[INFO] Using provided native topics for A from {native_topics_A_path}")
        A_native = load_topic_ids(native_topics_A_path, n_docs=len(docs_A))
    else:
        E_A_native = load_embeddings(embeddings_A_path, n_docs=len(docs_A), unit_norm=want_unit_A)
        print(f"[INFO] Assigning native A via "
              f"{'similarity' if prefer_similarity or not _has_umap_and_cluster(model_A) else 'transform'}")
        A_native = _safe_assign(
            model_A, docs_A, embedder_A, want_unit_A, batch_size, device,
            E_docs=E_A_native, prefer_similarity=prefer_similarity,
        )

    # --- Native B ---
    if native_topics_B_path:
        print(f"[INFO] Using provided native topics for B from {native_topics_B_path}")
        B_native = load_topic_ids(native_topics_B_path, n_docs=len(docs_B))
    else:
        E_B_native = load_embeddings(embeddings_B_path, n_docs=len(docs_B), unit_norm=want_unit_B)
        print(f"[INFO] Assigning native B via "
              f"{'similarity' if prefer_similarity or not _has_umap_and_cluster(model_B) else 'transform'}")
        B_native = _safe_assign(
            model_B, docs_B, embedder_B, want_unit_B, batch_size, device,
            E_docs=E_B_native, prefer_similarity=prefer_similarity,
        )

    # --- Cross A -> B ---
    print(f"[INFO] Cross-assign A -> B via "
          f"{'similarity' if prefer_similarity or not _has_umap_and_cluster(model_B) else 'transform'}")
    A_cross = _safe_assign(
        model_B, docs_A, embedder_B, want_unit_B, batch_size, device,
        E_docs=None, prefer_similarity=prefer_similarity,
    )

    # --- Cross B -> A ---
    print(f"[INFO] Cross-assign B -> A via "
          f"{'similarity' if prefer_similarity or not _has_umap_and_cluster(model_A) else 'transform'}")
    B_cross = _safe_assign(
        model_A, docs_B, embedder_A, want_unit_A, batch_size, device,
        E_docs=None, prefer_similarity=prefer_similarity,
    )

    return (
        np.asarray(A_native, dtype=np.int32),
        np.asarray(A_cross, dtype=np.int32),
        np.asarray(B_native, dtype=np.int32),
        np.asarray(B_cross, dtype=np.int32),
    )

In [ ]:
A_native, A_cross, B_native, B_cross = cross_assign_bt(
    model_A=model_A, docs_A=docs_A, embedder_A=corpus_A.embedder,
    train_emb_path_A=corpus_A.train_embeddings_path,
    native_topics_A_path=corpus_A.native_topics_path,
    embeddings_A_path=corpus_A.embeddings_path,
    model_B=model_B, docs_B=docs_B, embedder_B=corpus_B.embedder,
    train_emb_path_B=corpus_B.train_embeddings_path,
    native_topics_B_path=corpus_B.native_topics_path,
    embeddings_B_path=corpus_B.embeddings_path,
    batch_size=run.batch_size,
    device=run.device,
    prefer_similarity=run.prefer_similarity,
)

print(f"\nAssignment shapes:")
print(f"  A native: {A_native.shape}, A cross: {A_cross.shape}")
print(f"  B native: {B_native.shape}, B cross: {B_cross.shape}")

## Identify Matched Topic Pairs

Compute the pairing strength matrices and corpus-level alignment metrics:

- **Coverage (C)**: Average fraction of a native topic's documents that map to *any* non-outlier cross topic.
- **Uniqueness (U)**: Fraction mapping to the outlier topic -1 in the foreign model.
- **Alignment (A)**: Average best-match strength across native topics.

Weighted variants (Cw, Uw, Aw) account for topic size.

In [ ]:
def pairing_strength(native_topics: np.ndarray, cross_topics: np.ndarray):
    """Compute the pairing strength matrix S and per-topic document counts."""
    assert native_topics.shape == cross_topics.shape
    counts_native = Counter(native_topics)
    pair_counts = Counter(zip(native_topics, cross_topics))

    native_ids = sorted(counts_native.keys())
    cross_ids = sorted(set(cross_topics.tolist()))

    rows = []
    for ti in native_ids:
        denom = counts_native[ti]
        row = {tj: pair_counts.get((ti, tj), 0) / denom for tj in cross_ids}
        rows.append(pd.Series(row, name=ti))

    S = pd.DataFrame(rows).fillna(0.0)
    cols = [c for c in S.columns if c != -1] + ([-1] if -1 in S.columns else [])
    return S[cols], counts_native


def corpus_metrics(S: pd.DataFrame, counts_native: Counter) -> Dict[str, Union[float, pd.Series]]:
    """Compute corpus-level alignment metrics."""
    T = S.shape[0]
    weights = np.array([counts_native[i] for i in S.index])

    non_out = [c for c in S.columns if c != -1]
    out_col = -1 if -1 in S.columns else None

    C = S[non_out].to_numpy().sum() / T
    Cw = (weights[:, None] * S[non_out].to_numpy()).sum() / weights.sum()

    U_series = S[out_col] if out_col is not None else pd.Series(0.0, index=S.index)
    U = U_series.sum() / T
    Uw = (weights * U_series.to_numpy()).sum() / weights.sum()

    SA = S[non_out].max(axis=1)
    A = SA.mean()
    Aw = (weights * SA.to_numpy()).sum() / weights.sum()

    return {
        "C": C, "Cw": Cw, "theta": Cw - C,
        "U": U, "Uw": Uw,
        "A": A, "Aw": Aw, "Aw_minus_A": Aw - A,
        "SA": SA,
    }


def adjusted_metrics(S, counts_native):
    """Compute adjusted metrics normalised by non-outlier mass."""
    non_out = [c for c in S.columns if c != -1]
    out = S[-1] if -1 in S.columns else pd.Series(0.0, index=S.index)
    w = np.array([counts_native[i] for i in S.index])
    denom = (1.0 - out).replace(0, 1.0)
    Snorm = S[non_out].div(denom, axis=0).fillna(0.0)
    Cp = Snorm.to_numpy().sum() / S.shape[0]
    Cpw = (w[:, None] * Snorm.to_numpy()).sum() / w.sum()
    Ap = Snorm.max(axis=1).mean()
    Apw = (w * Snorm.max(axis=1).to_numpy()).sum() / w.sum()
    return {"C'": Cp, "Cw'": Cpw, "A'": Ap, "Aw'": Apw}


def top_matches(S, k=3):
    """Extract top-k cross-topic matches for each native topic."""
    out = []
    for ti, row in S.drop(columns=[-1], errors="ignore").iterrows():
        out.append((ti, row.sort_values(ascending=False).head(k)))
    rows = []
    for ti, s in out:
        for tj, val in s.items():
            rows.append({"topic_native": ti, "topic_cross": tj, "S": float(val)})
    return pd.DataFrame(rows)

In [ ]:
# Compute pairing strength matrices
S_A, counts_A = pairing_strength(A_native, A_cross)
S_B, counts_B = pairing_strength(B_native, B_cross)

# Compute metrics
metrics_A = corpus_metrics(S_A, counts_A)
metrics_B = corpus_metrics(S_B, counts_B)

print(f"=== Metrics for {corpus_A.name} (A -> B) ===")
for k, v in metrics_A.items():
    if k != "SA":
        print(f"  {k}: {v:.4f}")

print(f"\n=== Metrics for {corpus_B.name} (B -> A) ===")
for k, v in metrics_B.items():
    if k != "SA":
        print(f"  {k}: {v:.4f}")

In [ ]:
# Top matches for each direction
topA = top_matches(S_A, k=run.top_k_matches)
topB = top_matches(S_B, k=run.top_k_matches)

print(f"Top matches: {corpus_A.name} topics -> {corpus_B.name} topics")
display(topA.head(20))

print(f"\nTop matches: {corpus_B.name} topics -> {corpus_A.name} topics")
display(topB.head(20))

## Save Results

Persist all outputs: pairing strength matrices, corpus metrics, per-topic alignment scores, and top matches.

In [ ]:
os.makedirs(run.output_dir, exist_ok=True)

# Pairing strength matrices
S_A.to_csv(os.path.join(run.output_dir, f"BTM_S_{corpus_A.name}_vs_{corpus_B.name}.csv"))
S_B.to_csv(os.path.join(run.output_dir, f"BTM_S_{corpus_B.name}_vs_{corpus_A.name}.csv"))

# Scalar metrics
pd.Series(metrics_A).drop(labels=["SA"]).to_csv(
    os.path.join(run.output_dir, f"BTM_metrics_{corpus_A.name}.csv")
)
pd.Series(metrics_B).drop(labels=["SA"]).to_csv(
    os.path.join(run.output_dir, f"BTM_metrics_{corpus_B.name}.csv")
)

# Per-topic alignment scores with topic names
for corpus, model, metrics, counts, label in [
    (corpus_A, model_A, metrics_A, counts_A, corpus_A.name),
    (corpus_B, model_B, metrics_B, counts_B, corpus_B.name),
]:
    sa = metrics["SA"].rename("SA").to_frame()
    names = model.get_topic_info().set_index("Topic")["Name"]
    sa["topic_name"] = sa.index.map(names)
    sa["n_docs"] = sa.index.map(lambda t: counts.get(t, 0))
    sa.sort_values("SA", ascending=True).to_csv(
        os.path.join(run.output_dir, f"BTM_SA_{label}_by_topic.csv"),
        index_label="topic",
    )

# Adjusted metrics
pd.Series(adjusted_metrics(S_A, counts_A)).to_csv(
    os.path.join(run.output_dir, f"BTM_metrics_{corpus_A.name}_adjusted.csv")
)
pd.Series(adjusted_metrics(S_B, counts_B)).to_csv(
    os.path.join(run.output_dir, f"BTM_metrics_{corpus_B.name}_adjusted.csv")
)

# Top matches
topA.to_csv(os.path.join(run.output_dir, f"top_matches_{corpus_A.name}.csv"), index=False)
topB.to_csv(os.path.join(run.output_dir, f"top_matches_{corpus_B.name}.csv"), index=False)

print(f"All outputs saved to: {run.output_dir}")

---
<!-- nav-footer -->

← [03 topic ranking and narrowing](../../../../../SocialMedia/Reddit/notebooks/BERTopic/03_topic_refinement/03_topic_ranking_and_narrowing.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [02 finalizing pairs](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/02_finalizing_pairs.ipynb) →
